In [0]:
df = spark.read.table("workspace.finvizwebscrapping.01_bronze_layer")
display(df)



In [0]:
from pyspark.sql.functions import to_date, col
from delta.tables import DeltaTable

source_table = "workspace.finvizwebscrapping.01_bronze_layer"
target_table = "workspace.finvizwebscrapping.02_silver_layer"

# Read source data
source_df = spark.read.table(source_table)

# Add a date-only column for merge purposes
source_df = source_df.withColumn(
    "Ingestion_Day",
    to_date(col("Ingestion_Date"))
)

# Deduplicate source data, keeping the latest record per (Metrics, Ticker, Ingestion_Day)
from pyspark.sql import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("Metrics", "Ticker", "Ingestion_Day").orderBy(col("Ingestion_Date").desc())
source_df = source_df.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

# Create target table if it doesn't exist
if not spark.catalog.tableExists(target_table):

    (
        source_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

else:

    delta_target = DeltaTable.forName(spark, target_table)

    (
        delta_target.alias("t")
        .merge(
            source_df.alias("s"),
            """
            t.Metrics = s.Metrics
            AND t.Ticker = s.Ticker
            AND TO_DATE(t.Ingestion_Date) = s.Ingestion_Day
            """
        )
        .whenMatchedUpdate(
            set={
                "Metrics": "s.Metrics",
                "Value": "s.Value",
                "Ticker": "s.Ticker",
                "Ingestion_Date": "s.Ingestion_Date",
                "Ingestion_Day": "s.Ingestion_Day"
            }
        )
        .whenNotMatchedInsert(
            values={
                "Metrics": "s.Metrics",
                "Value": "s.Value",
                "Ticker": "s.Ticker",
                "Ingestion_Date": "s.Ingestion_Date",
                "Ingestion_Day": "s.Ingestion_Day"
            }
        )
        .execute()
    )

print("Upsert completed successfully.")


In [0]:
display(spark.read.table("workspace.finvizwebscrapping.02_silver_layer"))